In [ ]:
# Cài PySpark (tự lo luôn Spark + Java)
!pip install pyspark -q

# Khởi tạo SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Final") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("✅ Spark đã sẵn sàng!")

✅ Spark đã sẵn sàng!


In [ ]:
!pip install -q streamlit
print("\nStreamlit installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 48.8 MB/s eta 0:00:00

Streamlit installed successfully.


In [ ]:
!python --version

Python 3.12.13


In [ ]:
import gradio as gr
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.clustering import KMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.feature import VectorAssembler

# =========================
# INIT SPARK
# =========================
spark = SparkSession.builder \
    .appName("EcommerceApp") \
    .getOrCreate()

# =========================
# DASHBOARD FUNCTION
# =========================
def dashboard(file):
    if file is None:
        return "Chưa upload file", None

    df = pd.read_csv(file.name)
    info = f"Số dòng: {len(df)} | Số cột: {len(df.columns)}"
    return info, df.head()

# =========================
# PHÂN KHÚC KH
# =========================
def clustering(file):
    if file is None:
        return "Chưa upload file"

    df = pd.read_csv(file.name)
    sdf = spark.createDataFrame(df)

    assembler = VectorAssembler(
        inputCols=df.columns,
        outputCol="features"
    )
    data = assembler.transform(sdf)

    kmeans = KMeans(k=3)
    model = kmeans.fit(data)

    result = model.transform(data)
    return result.select("prediction").toPandas()

# =========================
# KHUYẾN NGHỊ
# =========================
def recommend(file, user_id):
    if file is None:
        return "Chưa upload file"

    df = pd.read_csv(file.name)
    sdf = spark.createDataFrame(df)

    als = ALS(
        userCol="customer_id",
        itemCol="product_id",
        ratingCol="rating",
        coldStartStrategy="drop"
    )

    model = als.fit(sdf)

    users = spark.createDataFrame([(int(user_id),)], ["customer_id"])
    recs = model.recommendForUserSubset(users, 10)

    return recs.toPandas()

# =========================
# FP-GROWTH
# =========================
def association(file):
    if file is None:
        return "Chưa upload file", None

    df = pd.read_csv(file.name)
    sdf = spark.createDataFrame(df)

    fp = FPGrowth(
        itemsCol="items",
        minSupport=0.2,
        minConfidence=0.6
    )

    model = fp.fit(sdf)

    return (
        model.freqItemsets.toPandas(),
        model.associationRules.toPandas()
    )

# =========================
# DỰ ĐOÁN
# =========================
def predict(price, quantity):
    return price * quantity

# =========================
# ADMIN
# =========================
def admin_upload(file):
    if file is None:
        return "Chưa có file"

    df = pd.read_csv(file.name)
    return df.head()

def retrain():
    return "Đã retrain thành công!"

def report():
    return "Đang tạo báo cáo..."

# =========================
# UI GRADIO
# =========================
with gr.Blocks() as app:
    gr.Markdown("# Hệ thống Phân tích Hành vi Khách hàng")

    with gr.Tabs():

        # ================= DASHBOARD =================
        with gr.Tab("Dashboard"):
            file_dash = gr.File()
            btn_dash = gr.Button("Phân tích")

            output_info = gr.Textbox(label="Thông tin")
            output_table = gr.Dataframe()

            btn_dash.click(
                dashboard,
                inputs=file_dash,
                outputs=[output_info, output_table]
            )

        # ================= CLUSTER =================
        with gr.Tab("Phân khúc KH"):
            file_cluster = gr.File()
            btn_cluster = gr.Button("Chạy KMeans")

            output_cluster = gr.Dataframe()

            btn_cluster.click(
                clustering,
                inputs=file_cluster,
                outputs=output_cluster
            )

        # ================= RECOMMEND =================
        with gr.Tab("Khuyến nghị"):
            file_rec = gr.File()
            user_id = gr.Number(label="Customer ID")

            btn_rec = gr.Button("Gợi ý")

            output_rec = gr.Dataframe()

            btn_rec.click(
                recommend,
                inputs=[file_rec, user_id],
                outputs=output_rec
            )

        # ================= FP-GROWTH =================
        with gr.Tab("Xu hướng mua sắm"):
            file_fp = gr.File()
            btn_fp = gr.Button("Phân tích")

            output_freq = gr.Dataframe(label="Frequent Itemsets")
            output_rules = gr.Dataframe(label="Association Rules")

            btn_fp.click(
                association,
                inputs=file_fp,
                outputs=[output_freq, output_rules]
            )

        # ================= PREDICT =================
        with gr.Tab("Dự đoán"):
            price = gr.Number(label="Giá")
            quantity = gr.Number(label="Số lượng")

            btn_pred = gr.Button("Dự đoán")
            output_pred = gr.Textbox()

            btn_pred.click(
                predict,
                inputs=[price, quantity],
                outputs=output_pred
            )

        # ================= ADMIN =================
        with gr.Tab("Admin"):
            file_admin = gr.File()
            btn_upload = gr.Button("Upload")

            output_admin = gr.Dataframe()

            btn_upload.click(
                admin_upload,
                inputs=file_admin,
                outputs=output_admin
            )

            btn_retrain = gr.Button("Retrain")
            output_retrain = gr.Textbox()

            btn_retrain.click(
                retrain,
                outputs=output_retrain
            )

            btn_report = gr.Button("Xuất báo cáo")
            output_report = gr.Textbox()

            btn_report.click(
                report,
                outputs=output_report
            )

# =========================
# RUN
# =========================
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2755b82c4ee713efa6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import streamlit as st
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.clustering import KMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.feature import VectorAssembler

# =========================
# INIT SPARK
# =========================
@st.cache_resource
def create_spark():
    return SparkSession.builder \
        .appName("EcommerceApp") \
        .getOrCreate()

spark = create_spark()

st.title("Hệ thống Phân tích Hành vi Khách hàng")

# =========================
# MENU
# =========================
menu = st.sidebar.selectbox("Chọn chức năng", [
    "Dashboard",
    "Phân khúc KH",
    "Khuyến nghị",
    "Xu hướng mua sắm",
    "Dự đoán",
    "Admin"
])

# =========================
# DASHBOARD
# =========================
if menu == "Dashboard":
    st.header("Dashboard")

    file = st.file_uploader("Upload dữ liệu", type=["csv"])
    if file:
        df = pd.read_csv(file)
        st.write(df.head())

        st.metric("Số dòng", len(df))
        st.metric("Số cột", len(df.columns))

        st.bar_chart(df.select_dtypes(include='number'))

# =========================
# PHÂN KHÚC KH
# =========================
elif menu == "Phân khúc KH":
    st.header("Phân khúc khách hàng")

    file = st.file_uploader("Upload CSV", type=["csv"])
    if file:
        df = pd.read_csv(file)
        sdf = spark.createDataFrame(df)

        assembler = VectorAssembler(
            inputCols=df.columns,
            outputCol="features"
        )
        data = assembler.transform(sdf)

        kmeans = KMeans(k=3)
        model = kmeans.fit(data)

        result = model.transform(data)
        st.write(result.select("prediction").toPandas())

# =========================
# KHUYẾN NGHỊ
# =========================
elif menu == "Khuyến nghị":
    st.header("Khuyến nghị sản phẩm")

    file = st.file_uploader("Upload dữ liệu rating", type=["csv"])
    if file:
        df = pd.read_csv(file)
        sdf = spark.createDataFrame(df)

        als = ALS(
            userCol="customer_id",
            itemCol="product_id",
            ratingCol="rating",
            coldStartStrategy="drop"
        )

        model = als.fit(sdf)

        user_id = st.number_input("Nhập customer_id", min_value=1)

        users = spark.createDataFrame([(user_id,)], ["customer_id"])
        recs = model.recommendForUserSubset(users, 10)

        st.write(recs.toPandas())

# =========================
# FP-GROWTH
# =========================
elif menu == "Xu hướng mua sắm":
    st.header("Association Rules")

    file = st.file_uploader("Upload basket data", type=["csv"])
    if file:
        df = pd.read_csv(file)
        sdf = spark.createDataFrame(df)

        fp = FPGrowth(
            itemsCol="items",
            minSupport=0.2,
            minConfidence=0.6
        )

        model = fp.fit(sdf)

        st.write("Frequent Itemsets")
        st.write(model.freqItemsets.toPandas())

        st.write("Association Rules")
        st.write(model.associationRules.toPandas())

# =========================
# DỰ ĐOÁN
# =========================
elif menu == "Dự đoán":
    st.header("Dự đoán")

    price = st.number_input("Giá")
    quantity = st.number_input("Số lượng")

    if st.button("Dự đoán"):
        prediction = price * quantity  # demo
        st.success(f"Giá trị dự đoán: {prediction}")

# =========================
# ADMIN
# =========================
elif menu == "Admin":
    st.header("Quản trị hệ thống")

    st.subheader("Upload dữ liệu")
    file = st.file_uploader("Upload data", type=["csv"])

    if file:
        df = pd.read_csv(file)
        st.write("Preview:", df.head())

    if st.button("Retrain model"):
        st.success("Đã retrain thành công!")

    if st.button("Xuất báo cáo"):
        st.info("Đang tạo báo cáo...")

2026-03-31 13:13:04.913 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:13:04.923 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:13:05.538 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-31 13:13:05.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:13:05.546 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:13:05.548 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:13:05.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn